# Gauge-identifiable C1 — where does the OOD failure live?

**Self-contained: upload this `.ipynb` to Colab and run all cells.** Nothing is trained here.

Notebook 11 found that C1's spectral-shift failure sits in the learned kinetic operator, but also
that the two learned halves are only defined **up to a gauge**: (κ, ν) and (κ − c, ν + c) give the
same C1, and the Phase 6 checkpoints carry κθ(0) ≈ −25. This notebook finishes that story:

| Section | Experiment | Needs |
|---|---|---|
| (a) | Reciprocal ablation stats on the saved run: K_exact+L_θ vs K_θ+L_exact | Drive run `3bf81deae4ef1ff9` |
| (b) | Port cross-check, then gauged swaps on spectral (G4) and β/V (G2, G3) axes | Phase 6 C1 checkpoints |
| (c) | Local law: is L_θ ≈ βρ − V once the gauge is fixed? | (b), plus C1g if trained |
| (d) | The same ablation on **C1g**, trained with κθ(0) = 0 from the start | C1g from `00_training` (`C1G_LAMBDAS=[0, 0.01]`) |

## Pre-registered expectations (written before the C1g results)

1. **The gauge fix is an exact reparameterization**, not a smaller model class: the local net's
   free bias absorbs any constant. C1g therefore tests *identifiability and optimization*, not
   expressivity. C1 and C1g share seeds, initialization, data order and the 40-epoch protocol.
2. **C1g is not expected to repair bandwidth-16 error of the full model.** The learned dispersion
   plateaus above |k| ≈ 9 because the training ICs have no energy there — a data-support limit
   that no gauge choice touches.
3. **C1g is expected to** (i) close the raw-vs-aligned gap, (ii) make both component swaps
   meaningful *without* any post-hoc correction, and (iii) possibly move the effective
   nonlinearity from ≈ 0.88β toward β.
4. **Double dissociation test.** If the kinetic half is what fails under spectral shift, then
   K_exact+L_θ should survive G4 while K_θ+L_exact fails; under β/V shift (G2, G3) the
   reverse pattern implicates the local half.

## 1. Settings

`SMOKE=True` is a plumbing check on tiny settings, never a result. Section (b) and (d) write
resumable runs under `OUTPUT_ROOT`; rerunning after a disconnect reuses completed units.

In [ ]:
from pathlib import Path
import os, sys, json, importlib.util

SOURCE_ROOT = os.environ.get("SPNO_SOURCE_ROOT", "")          # Phase 6 checkpoints (auto-located)
CHECKPOINT_ARCHIVE = os.environ.get("SPNO_CHECKPOINT_ARCHIVE", "")
SOURCE_CONFIG = os.environ.get("SPNO_SOURCE_CONFIG", "")      # optional JSON with original `data`
OUTPUT_ROOT = Path(os.environ.get("SPNO_OUTPUT_ROOT", "/content/drive/MyDrive/spno/gauge-study"))
HYBRID_RUN = Path(os.environ.get("SPNO_HYBRID_RUN",
                                 "/content/drive/MyDrive/spno/hybrid-ablation/3bf81deae4ef1ff9"))
WORKFLOW_ROOT = Path(os.environ.get("SPNO_WORKFLOW_ROOT", "/content/drive/MyDrive/spno/workflow"))
SMOKE = False
SETTINGS = {
    "training_seeds": [0, 1, 2], "probe_seeds": [1000, 1001, 1002, 1003, 1004],
    "batch": 16, "bandwidths": [4, 8, 12, 16, 24],
    "short_steps": 200, "long_steps": 2000, "stride": 50,
    "reference_substeps": 32, "reference_tolerance": 1e-4,
    "spatial_samples": 2, "spatial_tolerance": 1e-3, "tail_threshold": 1e-6,
    "device": "cpu", "threads": 2, "allow_budget_bound": True, "bootstrap_draws": 2000,
    "gauge": True,
    # Spectral axis (stresses K), beta/alpha and V axes (stress L), and the control.
    "case_names": ["G1-interpolation", "G2-extrapolation", "G3-potential-strong",
                   "G3-potential-short", "G4-bandwidth-4", "G4-bandwidth-8",
                   "G4-bandwidth-12", "G4-bandwidth-16", "G4-bandwidth-24"],
}
# Section (b)/(c) probes: one probe batch, as in the 2026-09-23 Colab cells.
PROBE = {"probe_seed": 1000, "batch": 16, "steps": 200, "reference_substeps": 64,
         "local_law_batch": 256}
if SMOKE:
    SETTINGS.update(training_seeds=[0], probe_seeds=[1000, 1001], batch=2, short_steps=2,
                    long_steps=4, stride=1, reference_substeps=2, spatial_samples=1,
                    bootstrap_draws=50, smoke=True)
    PROBE.update(batch=2, steps=3, reference_substeps=4, local_law_batch=8)
SETTINGS.update(json.loads(os.environ.get("SPNO_OPTIONS", "{}")))
PROBE.update(json.loads(os.environ.get("SPNO_PROBE", "{}")))
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
print(json.dumps({"settings": SETTINGS, "probe": PROBE}, indent=2))

## 2. Install the embedded code and locate the Phase 6 checkpoints

In [ ]:
import base64, hashlib, io, subprocess, zipfile
from pathlib import PurePosixPath

EMBEDDED_SOURCE_SHA256 = "5659f18f40dd904af0fcec59197719b4cd45919ec86f8fea8b4fad50e99f4ae4"
EMBEDDED_SOURCE = """
..."""

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    for member in archive.infolist():
        name = PurePosixPath(member.filename)
        if name.is_absolute() or ".." in name.parts or "\\" in member.filename:
            raise ValueError("Unsafe archive member: " + member.filename)
        if not (destination / member.filename).resolve().is_relative_to(destination):
            raise ValueError("Archive member escapes destination")
    archive.extractall(destination)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
local_project = os.environ.get("SPNO_PROJECT_ROOT")
if local_project:
    PROJECT_ROOT = Path(local_project)
else:
    raw = base64.b64decode(EMBEDDED_SOURCE)
    assert hashlib.sha256(raw).hexdigest() == EMBEDDED_SOURCE_SHA256
    base = Path("/content") if IN_COLAB else Path.cwd()
    PROJECT_ROOT = base / ("spno-hybrid-code-" + EMBEDDED_SOURCE_SHA256[:12])
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        assert archive.testzip() is None
        safe_extract(archive, PROJECT_ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

if not SOURCE_ROOT:
    existing = Path("/content/pin/spno/results/phase6-standalone-artifacts")
    if existing.is_dir():
        SOURCE_ROOT = existing
    else:
        if not CHECKPOINT_ARCHIVE:
            drive_root = Path("/content/drive/MyDrive")
            names = ["phase6-eval-only-bd4e108527-K0.zip", "phase6-standalone-artifacts-bd4e108527-K0.zip"]
            hits = [p for name in names for p in drive_root.rglob(name)]
            if not hits:
                raise FileNotFoundError("Place the Phase 6 artifact ZIP in MyDrive, or set SOURCE_ROOT / CHECKPOINT_ARCHIVE in cell 1.")
            CHECKPOINT_ARCHIVE = str(sorted(hits)[0])
        archive_path = Path(CHECKPOINT_ARCHIVE)
        archive_digest = hashlib.sha256()
        with archive_path.open("rb") as stream:
            for block in iter(lambda: stream.read(8 << 20), b""):
                archive_digest.update(block)
        known = {
            "phase6-eval-only-bd4e108527-K0.zip": "cc882810f6fec63f36d6923833c05c6dcde5b6d50b2a1df296634be755b1235d",
            "phase6-standalone-artifacts-bd4e108527-K0.zip": "01fd894349dc36dd90386d877693e51bbe3d2d28e98609ccefdf02608a0a0812",
        }
        if archive_path.name in known:
            assert archive_digest.hexdigest() == known[archive_path.name], "Checkpoint archive checksum mismatch"
        extracted = PROJECT_ROOT.parent / ("spno-hybrid-artifacts-" + archive_digest.hexdigest()[:12])
        with zipfile.ZipFile(archive_path) as archive:
            assert archive.testzip() is None, "Corrupt checkpoint archive"
            safe_extract(archive, extracted)
        candidates = [p.parent for p in extracted.rglob("checkpoints") if (p / "phase6").is_dir()]
        assert len(candidates) == 1, f"Expected one artifact root, found {candidates}"
        SOURCE_ROOT = candidates[0]
SOURCE_ROOT = Path(SOURCE_ROOT)
assert (SOURCE_ROOT / "checkpoints" / "phase6").is_dir(), SOURCE_ROOT
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Embedded source:", EMBEDDED_SOURCE_SHA256)
print("Checkpoint root:", SOURCE_ROOT)
print("Results:", OUTPUT_ROOT)

## (a) Reciprocal ablation on the saved G1–G9 run — no new rollouts

Run `3bf81deae4ef1ff9` already saved K_θ+L_exact next to K_exact+L_θ, but its summary only paired
the hybrid with C1. `resummarize` rebuilds the summary from the saved units **without** a new
run identity; the previous `summary.json` is kept as `summary.v1.json`.

These swaps are **raw** (no gauge fix), so K_θ+L_exact carries the full κθ(0) ≈ −25 offset as a
global phase. Read its raw error as "the gauge is wrong", not "the kinetic shape is wrong";
the aligned error and section (b) separate the two.

In [ ]:
import html
from IPython.display import display, Image, HTML
from scripts.run_hybrid_ablation import (resummarize, run_study, load_c1_cohorts,
                                        load_c1g_cohort, clean_json)
from scripts.plot_hybrid_ablation import export_plots
from spno.artifacts import atomic_json

def reciprocal_table(summary, metric="state_error", cohort="base"):
    rows = [r for r in summary["paired"] if r["endpoint"] == "final" and r["metric"] == metric
            and r["cohort"] == cohort and "model" in r]
    header = "<tr><th>case</th><th>model / baseline</th><th>ratio (geo. mean)</th><th>95% interval</th><th>status</th></tr>"
    body = ""
    for r in rows:
        ratio = r["ratio"]
        mean = "—" if ratio["mean"] is None else f"{ratio['mean']:.3g}"
        interval = "—" if ratio["low"] is None else f"[{ratio['low']:.3g}, {ratio['high']:.3g}]"
        body += (f"<tr><td>{html.escape(r['case'])}</td><td>{html.escape(r['model'])} / "
                 f"{html.escape(r['baseline'])}</td><td>{mean}</td><td>{interval}</td>"
                 f"<td>{html.escape(r['status'])}</td></tr>")
    caption = (f"<p>Final {metric.replace('_', ' ')}, paired ratio; &lt; 1 favors the first model. "
               "Descriptive intervals, not multiplicity-adjusted.</p>")
    return caption + "<table>" + header + body + "</table>"

def show_run(run, label):
    summary = json.loads((run / "summary.json").read_text())
    export_plots(run)
    print(label)
    for metric in ("state_error", "aligned_state_error"):
        display(HTML(reciprocal_table(summary, metric)))
    figure = run / "figures" / "07_reciprocal_ablation.png"
    if figure.exists():
        display(Image(filename=str(figure)))
    return summary

if (HYBRID_RUN / "manifest.json").exists():
    resummarize(HYBRID_RUN)
    saved_summary = show_run(HYBRID_RUN, f"(a) saved run {HYBRID_RUN.name}")
else:
    print(f"(a) skipped: no saved run at {HYBRID_RUN}")

## (b) Port cross-check, then gauged swaps on C1

The gauge-fixed swaps and the drift/δq probes were first run as ad hoc Colab cells on
2026-09-23; they now live in `spno.evaluation.component_ablation`. **Before trusting any new
number, this cell must reproduce the recorded ones:**

| Check | Recorded 2026-09-23 |
|---|---|
| Gauge move on full C1 | identical to C1 at 1e-12 |
| Gauge-fixed hybrid, bw 8, 200 steps | raw ≈ 6e-2, aligned ≈ 4e-3 |
| Gauge-fixed hybrid, bw 16 | aligned ≈ 5.4e-3 (C1: 0.88) |
| Drift predictor Σ dt⟨δq⟩_ρ | corr 1.000, slope 1.000, residual ≤ 0.3% |
| δq(ρ) slope | ≈ −0.12β on every seed (effective β ≈ 0.88β) |

In [ ]:
import torch, numpy as np
from spno.config import DataConfig
from spno.evaluation.component_ablation import (
    ComponentSplitStep, component_models, delta_q_profile, local_law, predicted_phase_drift, probe_cases, sample_probe)
from spno.solvers.split_step import SubsteppedReference

torch.set_num_threads(SETTINGS["threads"])
declared = DataConfig(**json.loads(Path(SOURCE_CONFIG).read_text())["data"]) if SOURCE_CONFIG else None
data_config, c1_cohorts, _ = load_c1_cohorts(SOURCE_ROOT, declared, SETTINGS["training_seeds"],
                                             allow_budget_bound=SETTINGS["allow_budget_bound"])
dt = data_config.dt
bands = [b for b in (8, 12, 16) if b <= data_config.grid_size // 2] or [data_config.initial_bandwidth]
cases = {c.name: c for c in probe_cases(data_config, bands)}
check_names = ["G1-interpolation"] + [f"G4-bandwidth-{b}" for b in bands]

def errors(pred, ref):
    raw = ((pred - ref).norm(dim=-1) / ref.norm(dim=-1)).mean().item()
    inner = (ref.conj() * pred).sum(-1)
    aligned = pred * (inner / inner.abs()).conj().unsqueeze(-1)
    return raw, ((aligned - ref).norm(dim=-1) / ref.norm(dim=-1)).mean().item()

port_check = {}
rho_samples = sample_probe(cases["G1-interpolation"], PROBE["probe_seed"],
                           PROBE["local_law_batch"])[0][0].abs().square().numpy()
with torch.inference_mode():
    for seed, c1 in c1_cohorts["base"].items():
        record = port_check.setdefault(str(seed), {})
        parts = component_models(c1, gauge=True)
        both = ComponentSplitStep(c1, exact_kinetic=False, exact_local=False, gauge="zero_mode")
        inputs, _ = sample_probe(cases["G1-interpolation"], PROBE["probe_seed"], PROBE["batch"])
        identity = (both(*inputs, dt) - c1(*inputs, dt)).abs().max().item()
        assert identity < 1e-12, identity
        record["identity_max_abs"] = identity
        for name in check_names:
            cfg = cases[name].config
            inputs, _ = sample_probe(cases[name], PROBE["probe_seed"], PROBE["batch"])
            x, V, a, b = inputs
            ops = {"reference": SubsteppedReference(cfg.domain, PROBE["reference_substeps"]), "C1": c1,
                   "raw hybrid": parts["exactK_learnedL"], "gauge-fixed hybrid": parts["exactK_learnedL_g"],
                   "raw reverse": parts["learnedK_exactL"], "gauge-fixed reverse": parts["learnedK_exactL_g"]}
            state = {k: x.clone() for k in ops}
            for _ in range(PROBE["steps"]):
                state = {k: op(state[k], V, a, b, cfg.dt) for k, op in ops.items()}
            for k in list(ops)[1:]:
                raw, al = errors(state[k], state["reference"])
                record.setdefault(name, {})[k] = {"raw": raw, "aligned": al}
                print(f"seed {seed} {name:18s} {k:20s} raw={raw:.3e} aligned={al:.3e}")
        drift_case = cases[check_names[1] if len(check_names) > 1 else check_names[0]]
        inputs, _ = sample_probe(drift_case, PROBE["probe_seed"], PROBE["batch"])
        drift = predicted_phase_drift(c1, inputs, drift_case.config.dt, steps=PROBE["steps"],
                                      reference_substeps=PROBE["reference_substeps"])
        record["drift"] = drift["stats"]
        for label, stats in drift["stats"].items():
            print(f"seed {seed} drift [{label:13s}] corr={stats['corr']:+.3f} slope={stats['slope']:.3f} "
                  f"residual={stats['relative_residual']:.3f}")
        profile = delta_q_profile(c1, rho_samples, shift=True)
        record["delta_q"] = profile["fits"]
        print(f"seed {seed} delta-q slope/beta:",
              ", ".join(f"β={f['beta']:+.1f}: {f['slope_over_beta']:+.3f}" for f in profile["fits"]
                        if f["slope_over_beta"] is not None))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
atomic_json(OUTPUT_ROOT / "port_check.json", clean_json(port_check))

### Gauged swaps on the selected axes (new resumable run)

Same frozen C1 checkpoints, now with **six** operators per case: C1, both raw swaps, both
gauge-fixed swaps, and the exact split step. The 2026-09-23 result predicts that the gauged
K_θ+L_exact loses its huge raw error while keeping the kinetic *shape* error at high bandwidth.

In [ ]:
C1_RUN = run_study(SOURCE_ROOT, OUTPUT_ROOT / "c1", data=data_config, options=SETTINGS)
c1_summary = show_run(C1_RUN, f"(b) C1 gauged swaps: {C1_RUN}")

## (c) Local law — is L_θ ≈ βρ − V?

Fit, on every grid point of one G1 probe batch (the data distribution), with ρ weights
(the global-phase weighting):
ν ≈ c_βρ·(βρ) + c_V·V + c₀ + c_α·(α − ᾱ). The truth is (1, −1, 0, 0).

* **C1 raw** — the local net as trained; its intercept carries the gauge constant.
* **C1 shifted** — ν + κθ(0), the post-hoc gauge fix. The intercept is removed by construction,
  so the informative comparison is **C1 shifted vs C1g** on c_βρ and c_V.
* **C1g** — trained with κθ(0) = 0; no shift exists to apply.

In [ ]:
import matplotlib.pyplot as plt
try:
    c1g_cohorts = load_c1g_cohort(WORKFLOW_ROOT, data_config, SETTINGS["training_seeds"])
except ValueError as error:
    c1g_cohorts = None
    print("C1g not available yet:", error)
    print("Train it in 00_training.ipynb: TRAIN_PHASES=[7], C1G_LAMBDAS=[0.0, 0.01].")

law_inputs, _ = sample_probe(cases["G1-interpolation"], PROBE["probe_seed"], PROBE["local_law_batch"])
variants = {"C1 raw": (c1_cohorts["base"], False), "C1 shifted": (c1_cohorts["base"], True)}
if c1g_cohorts:
    variants["C1g"] = (c1g_cohorts[0]["base"], False)
laws, profiles = {}, {}
with torch.inference_mode():
    for label, (by_seed, shift) in variants.items():
        for seed, model in by_seed.items():
            laws.setdefault(label, {})[str(seed)] = local_law(model, law_inputs, shift=shift)
            profiles.setdefault(label, {})[str(seed)] = delta_q_profile(model, rho_samples, shift=shift)
keys = ("beta_rho", "potential", "intercept", "alpha", "relative_rmse_vs_truth", "unexplained_relative_rmse")
header = "<tr><th>variant</th><th>seed</th>" + "".join(f"<th>{k}</th>" for k in keys) + "</tr>"
body = "".join(f"<tr><td>{label}</td><td>{seed}</td>" + "".join(f"<td>{law[k]:+.4f}</td>" for k in keys) + "</tr>"
               for label, by_seed in laws.items() for seed, law in by_seed.items())
display(HTML("<p>Truth: beta_rho=+1, potential=−1, intercept=0, alpha=0.</p><table>" + header + body + "</table>"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), layout="constrained")
colors = {"C1 raw": "#bf4b45", "C1 shifted": "#e39a8f", "C1g": "#21866c"}
for label, by_seed in profiles.items():
    if label == "C1 raw":
        continue  # its intercept (the gauge constant) dwarfs the shape
    for number, (seed, profile) in enumerate(by_seed.items()):
        middle = len(profile["fits"]) // 2 + 1  # a nonzero beta
        axes[0].plot(profile["rho_grid"], profile["curves"][middle], color=colors[label],
                     label=f"{label} (β={profile['fits'][middle]['beta']:+.1f})" if number == 0 else None)
axes[0].axhline(0, color="k", lw=.8)
axes[0].set(xlabel="Density ρ", ylabel="δq = learned − βρ at V=0, α=0.9", title="Local-rate bias")
labels = list(laws)
for index, key in enumerate(("beta_rho", "potential")):
    means = [np.mean([law[key] for law in laws[l].values()]) for l in labels]
    spread = [np.ptp([law[key] for law in laws[l].values()]) / 2 for l in labels]
    axes[1].bar(np.arange(len(labels)) + .38 * index, means, .36, yerr=spread, capsize=3,
                label={"beta_rho": "c_βρ (truth +1)", "potential": "c_V (truth −1)"}[key])
axes[1].axhline(1, color="#555", ls=":"); axes[1].axhline(-1, color="#555", ls=":")
axes[1].set_xticks(np.arange(len(labels)) + .19, labels)
axes[1].set(title="Fitted local law (mean ± half-range over seeds)")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=.2)
fig.savefig(OUTPUT_ROOT / "local_law.png", dpi=150)
plt.show()
atomic_json(OUTPUT_ROOT / "local_law.json", clean_json({"laws": laws, "profiles": profiles}))

## (d) The same ablation on C1g

Pre-registered: K_exact+L_θ and K_θ+L_exact of C1g should behave like the **gauged** swaps of C1
without any correction (the gauged and raw swaps coincide, since κθ(0) = 0 already), and the
hybrid's raw error should sit close to its aligned error. The full-model error at bandwidth 16 is
**not** expected to improve (data-support limit). G6a/G7 rows are absent: those cohorts were not
retrained.

In [ ]:
if c1g_cohorts:
    C1G_RUN = run_study(SOURCE_ROOT, OUTPUT_ROOT / "c1g", data=data_config, options=SETTINGS,
                        cohorts=c1g_cohorts)
    c1g_summary = show_run(C1G_RUN, f"(d) C1g: {C1G_RUN}")
    with torch.inference_mode():
        for seed, model in c1g_cohorts[0]["base"].items():
            drift_case = cases[check_names[1] if len(check_names) > 1 else check_names[0]]
            inputs, _ = sample_probe(drift_case, PROBE["probe_seed"], PROBE["batch"])
            stats = predicted_phase_drift(model, inputs, drift_case.config.dt, steps=PROBE["steps"],
                                          reference_substeps=PROBE["reference_substeps"])["stats"]
            print(f"C1g seed {seed} drift:", {k: round(v["slope"], 3) for k, v in stats.items()})
else:
    print("(d) skipped: train C1g first (see section (c)).")

## Reading the results

* **Localization claim** — needs the dissociation: G4 hurts the swaps that keep K_θ, G2/G3 hurt
  the swaps that keep L_θ, in the *gauged* rows. Raw-row differences alone can be pure gauge.
* **Gauge claim** — C1 vs C1g full-model rows are a paired comparison (same seeds, init, data,
  protocol). A C1g win on raw error with an unchanged aligned error means the gauge was costing
  phase, not shape.
* **Local-law claim** — compare C1 shifted with C1g on c_βρ. If C1g moves toward 1, gauge
  ambiguity was also hurting the *learning* of the nonlinearity; if both stay near 0.88, the
  bias is an L0 optimization limit (the L1/L2 rungs are the follow-up).
* All source checkpoints are budget-bound (40 epochs): every number here is **exploratory**.